In [4]:
# =====================================================
# 01_data_collection.ipynb
# Ironhack Final Project: AI High-Growth Stock Selector
# Author: Dr. Arjola ARAPI-GJINI
# Date: April 2026
# =====================================================

import yfinance as yf
import pandas as pd
import numpy as np
import time
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
import yfinance as yf
import pandas as pd

# Single stock example
ticker = yf.Ticker("NVDA")        # or "TSLA", "AAPL", etc.
 
# Historical prices (5 years)
history = ticker.history(period="5y")
print(history.head())

# Fundamentals
print(ticker.info)                # dictionary with 100+ fields (PE, growth rates, etc.)
print(ticker.financials)          # income statement
print(ticker.balance_sheet)
print(ticker.cashflow)
print(ticker.earnings)

# Multiple stocks at once
tickers = ["NVDA", "TSLA", "AMD", "SMCI", "ARM"]  # high-growth candidates
data = yf.download(tickers, period="5y")

                                Open       High        Low      Close  \
Date                                                                    
2021-04-12 00:00:00-04:00  14.253823  15.313901  14.103701  15.170761   
2021-04-13 00:00:00-04:00  15.193206  15.660528  15.087722  15.640079   
2021-04-14 00:00:00-04:00  15.585716  15.680976  15.189215  15.238591   
2021-04-15 00:00:00-04:00  15.623123  16.173486  15.592201  16.096680   
2021-04-16 00:00:00-04:00  16.012642  16.125109  15.825364  15.872496   

                              Volume  Dividends  Stock Splits  
Date                                                           
2021-04-12 00:00:00-04:00  869324000        0.0           0.0  
2021-04-13 00:00:00-04:00  676212000        0.0           0.0  
2021-04-14 00:00:00-04:00  385500000        0.0           0.0  
2021-04-15 00:00:00-04:00  598480000        0.0           0.0  
2021-04-16 00:00:00-04:00  335208000        0.0           0.0  
{'address1': '2788 San Tomas Expressway'

[*********************100%***********************]  5 of 5 completed


In [2]:
!pip install lxml tqdm

print("✅ lxml and tqdm installed successfully!")

✅ lxml and tqdm installed successfully!


In [6]:
# =====================================================
# 01_data_collection.ipynb - Part 1: Configuration & Ticker List
# =====================================================
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# ------------------- CONFIG -------------------
DATA_DIR = "data"
RAW_DIR = f"{DATA_DIR}/raw"
PROCESSED_DIR = f"{DATA_DIR}/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

NUM_STOCKS = 180   # Good balance - not too big, not too small

print("🚀 Starting data collection for high-growth stock prediction project...\n")

# ------------------- Get S&P 500 Tickers -------------------
print("Downloading S&P 500 tickers...")

try:
    url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
    sp500 = pd.read_csv(url)
    sp500_tickers = sp500['Symbol'].tolist()
    print(f"✅ Loaded {len(sp500_tickers)} S&P 500 tickers from CSV.")
except:
    print("CSV method failed, using fallback...")
    sp500_tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AVGO", "LLY", "JPM"]

# ------------------- High-Growth Stocks (My Original Lis  + New Ones) -------------------
high_growth_seeds = [
    # Your original list
    "NVDA", "TSLA", "AMD", "SMCI", "ARM", "AVGO", "ASML", "PLTR", "CRWD",
    "NOW", "PANW", "SNOW", "DDOG", "MU", "AMAT", "KLAC", "LLY", "VRTX",
    
    # New additions - AI / Software / Infrastructure
    "NET", "PATH", "APP", "VRT",
    
    # Semiconductors / AI Chips / Robotics
    "LITE", "CRDO", "SYM",
    
    # Biotech / Pharma
    "REGN", "TERN", "ADMA", "ANIP",
    
    # Other high-potential / newer names
    "RKLB", "ASTS", "SOUN", "UPST", "IONQ", "QBTS", "PGY", "CELH"
]

# Remove duplicates and combine with S&P 500
all_tickers = list(set(sp500_tickers + high_growth_seeds))
np.random.shuffle(all_tickers)
selected_tickers = all_tickers[:NUM_STOCKS]

print(f"✅ Final selection: {len(selected_tickers)} stocks")
print(f"Including {len(high_growth_seeds)} high-growth and high-potential stocks")
print(f"Sample tickers: {selected_tickers[:15]}")
print("\nReady for downloading price and fundamental data...\n")

🚀 Starting data collection for high-growth stock prediction project...

✅ Loaded 503 S&P 500 tickers from CSV.
✅ Final selection: 180 stocks
Including 37 high-growth and high-potential stocks
Sample tickers: ['MRNA', 'TMUS', 'FANG', 'ELV', 'MAS', 'ETN', 'VLTO', 'VLO', 'KMB', 'PLD', 'ADP', 'TTD', 'WSM', 'SYK', 'FITB']

Ready for downloading price and fundamental data...



In [7]:
print("📈 Downloading historical price data... This may take 2-5 minutes.")

price_data = yf.download(
    tickers=selected_tickers,
    start="2015-01-01",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    threads=True
)

price_data.to_pickle(f"{RAW_DIR}/historical_prices_multi.pkl")
print(f"✅ Historical prices saved to: {RAW_DIR}/historical_prices_multi.pkl")

📈 Downloading historical price data... This may take 2-5 minutes.


[************          24%                       ]  43 of 180 completed$BRK.B: possibly delisted; no timezone found
[*********************100%***********************]  180 of 180 completed

1 Failed download:
['BRK.B']: possibly delisted; no timezone found


✅ Historical prices saved to: data/raw/historical_prices_multi.pkl


In [8]:
# CELL 4: Download Fundamentals

print("📊 Downloading company fundamentals... This may take 8-15 minutes.")

fundamentals_list = []

for i, ticker in enumerate(tqdm(selected_tickers, desc="Fetching fundamentals")):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        row = {
            'ticker': ticker,
            'company_name': info.get('longName'),
            'sector': info.get('sector'),
            'industry': info.get('industry'),
            'market_cap': info.get('marketCap'),
            'current_price': info.get('currentPrice'),
            'forward_pe': info.get('forwardPE'),
            'trailing_pe': info.get('trailingPE'),
            'peg_ratio': info.get('pegRatio'),
            'revenue_growth': info.get('revenueGrowth'),
            'earnings_growth': info.get('earningsGrowth'),
            'return_on_equity': info.get('returnOnEquity'),
            'debt_to_equity': info.get('debtToEquity'),
            'beta': info.get('beta'),
        }
        fundamentals_list.append(row)
        
        time.sleep(0.5)   # Could be changed
        
    except Exception as e:
        fundamentals_list.append({'ticker': ticker, 'error': str(e)})

fundamentals_df = pd.DataFrame(fundamentals_list)
fundamentals_df.to_csv(f"{PROCESSED_DIR}/fundamentals.csv", index=False)
fundamentals_df.to_pickle(f"{PROCESSED_DIR}/fundamentals.pkl")

print(f"\n✅ Done! Fundamentals saved for {len(fundamentals_df)} stocks.")
print(f"File location: {PROCESSED_DIR}/fundamentals.csv")

📊 Downloading company fundamentals... This may take 8-15 minutes.


Fetching fundamentals: 100%|██████████████████| 180/180 [03:15<00:00,  1.09s/it]


✅ Done! Fundamentals saved for 180 stocks.
File location: data/processed/fundamentals.csv
